In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR2TOT_H.xpt"
csv_file = DATA_DIR / "DR2TOT_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Keep untouched source for fidelity comparison.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 DR2TOT_H XPT -> CSV")
print("=" * 90)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTDRD1",
    "WTDR2D",
    "DR2DRSTZ",
    "DR2EXMER",
    "DRABF",
    "DRDINT",
    "DR2DBIH",
    "DR2DAY",
    "DR2LANG",
    "DR2MNRSP",
    "DR2HELPD",
    "DR2TNUMF",
    "DR2STY",
    "DR2SKY",
    "DR2TKCAL",
    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",
    "DR2TCHOL",
    "DR2TATOC",
    "DR2TATOA",
    "DR2TRET",
    "DR2TVARA",
    "DR2TACAR",
    "DR2TBCAR",
    "DR2TCRYP",
    "DR2TLYCO",
    "DR2TLZ",
    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",
    "DR2TFOLA",
    "DR2TFA",
    "DR2TFF",
    "DR2TFDFE",
    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",
    "DR2TCALC",
    "DR2TPHOS",
    "DR2TMAGN",
    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSODI",
    "DR2TPOTA",
    "DR2TSELE",
    "DR2TCAFF",
    "DR2TTHEO",
    "DR2TALCO",
    "DR2TMOIS",
    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",
    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",
    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",
    "DR2_300",
    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ",
    "DR2TWS"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DR2TOT_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9813:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 85:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,813 rows x 85 columns."
)


# ============================================================
# 4. DEFINE GENUINE DIETARY DECIMAL COLUMNS
# ============================================================

#
# CRITICAL:
#
# These 50 fields contain genuine fractional values.
#
# DO NOT:
#
#   - round them
#   - convert them to Int64
#   - strip '.0' globally
#   - use float_format in to_csv()
#

DECIMAL_COLUMNS = [
    "WTDRD1",
    "WTDR2D",
    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",
    "DR2TATOC",
    "DR2TATOA",
    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",
    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",
    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSELE",
    "DR2TALCO",
    "DR2TMOIS",
    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",
    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",
    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",
    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 50:

    raise ValueError(
        "Expected exactly 50 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 35:

    raise ValueError(
        "Expected exactly 35 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine Dietary decimal fields:",
    len(DECIMAL_COLUMNS)
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if len(numeric_cols) != 85:

    raise ValueError(
        "Expected all 85 variables to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact pattern from attached DR2TOT_H.xpt.
#
# 50,469 artifacts across 72 variables.
#

EXPECTED_TINY_COUNTS = {

    "WTDRD1": 1152,
    "WTDR2D": 1087,

    "DR2DBIH": 8,
    "DR2TNUMF": 1,

    "DR2TKCAL": 7,
    "DR2TPROT": 10,
    "DR2TCARB": 8,
    "DR2TSUGR": 11,
    "DR2TFIBE": 85,
    "DR2TTFAT": 9,
    "DR2TSFAT": 10,
    "DR2TMFAT": 10,
    "DR2TPFAT": 10,
    "DR2TCHOL": 108,
    "DR2TATOC": 9,
    "DR2TATOA": 6094,
    "DR2TRET": 82,
    "DR2TVARA": 23,
    "DR2TACAR": 1086,
    "DR2TBCAR": 119,
    "DR2TCRYP": 820,
    "DR2TLYCO": 2118,
    "DR2TLZ": 92,
    "DR2TVB1": 9,
    "DR2TVB2": 10,
    "DR2TNIAC": 8,
    "DR2TVB6": 10,
    "DR2TFOLA": 10,
    "DR2TFA": 260,
    "DR2TFF": 74,
    "DR2TFDFE": 10,
    "DR2TCHL": 9,
    "DR2TVB12": 33,
    "DR2TB12A": 4737,
    "DR2TVC": 51,
    "DR2TVD": 212,
    "DR2TVK": 15,
    "DR2TCALC": 1,
    "DR2TPHOS": 9,
    "DR2TMAGN": 3,
    "DR2TIRON": 9,
    "DR2TZINC": 4,
    "DR2TCOPP": 2,
    "DR2TSODI": 2,
    "DR2TPOTA": 8,
    "DR2TSELE": 9,
    "DR2TCAFF": 2242,
    "DR2TTHEO": 4253,
    "DR2TALCO": 6721,
    "DR2TMOIS": 1,

    "DR2TS040": 365,
    "DR2TS060": 400,
    "DR2TS080": 120,
    "DR2TS100": 44,
    "DR2TS120": 27,
    "DR2TS140": 15,
    "DR2TS160": 10,
    "DR2TS180": 12,

    "DR2TM161": 37,
    "DR2TM181": 10,
    "DR2TM201": 54,
    "DR2TM221": 1289,

    "DR2TP182": 10,
    "DR2TP183": 11,
    "DR2TP184": 3548,
    "DR2TP204": 99,
    "DR2TP205": 912,
    "DR2TP225": 673,
    "DR2TP226": 1104,

    "DR2_320Z": 1504,
    "DR2_330Z": 3879,
    "DR2BWATZ": 4675
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DR2TOT_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if len(EXPECTED_TINY_COUNTS) != 72:

    raise ValueError(
        "Expected exactly 72 variables "
        "containing tiny artifacts."
    )


if tiny_total != 50469:

    raise ValueError(
        f"Unexpected total tiny count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 50,469-value tiny-zero "
    "pattern across 72 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# NEVER do:
#
#     df.replace(TINY_VALUE, 0)
#
# blindly over the complete dataframe.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 50469:

    raise ValueError(
        "Incorrect total restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# Used later for exact numerical fidelity testing.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# np.round() below is ONLY used as a TEST.
#
# It does NOT modify any dietary values.
#

print("\n--- GENUINE DECIMAL DETECTION ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Genuine decimal-field pattern changed.\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Detected:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 50 genuine Dietary "
    "decimal variables detected."
)


# ============================================================
# 11. DECIMAL FIELD INFORMATION
# ============================================================

print("\n--- DECIMAL FIELD DETAILS ---")


for col in DECIMAL_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    print(
        f"{col:10s} | "
        f"nonmissing={len(values):5,d} | "
        f"fractional={int(fractional_mask.sum()):5,d} | "
        f"whole-looking={int((~fractional_mask).sum()):5,d}"
    )


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractions."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 13. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# 35 fields are proven integer/code fields.
#
# The 50 genuine decimal fields remain untouched.
#
# NO .round() is used for conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 35 whole-number/code fields "
    "converted to nullable Int64."
)


print(
    "PASS: 50 genuine Dietary decimal "
    "fields remain untouched."
)


# ============================================================
# 14. CDC CHECK - WTDRD1
# ============================================================

#
# Day 1 dietary sample weight:
#
# >0 = 8,661
#  0 = 1,152
# missing = 0
#

s = df["WTDRD1"]


if not (
    int(s.gt(0).sum()) == 8661
    and
    int(s.eq(0).sum()) == 1152
    and
    int(s.isna().sum()) == 0
):

    raise ValueError(
        "WTDRD1 differs from CDC."
    )


print(
    "PASS: WTDRD1 = "
    "8,661 positive + 1,152 zero."
)


# ============================================================
# 15. CDC CHECK - WTDR2D
# ============================================================

#
# Two-day sample weight:
#
# >0      = 7,574
# 0       = 1,087
# missing = 1,152
#

s = df["WTDR2D"]


if not (
    int(s.gt(0).sum()) == 7574
    and
    int(s.eq(0).sum()) == 1087
    and
    int(s.isna().sum()) == 1152
):

    raise ValueError(
        "WTDR2D differs from CDC."
    )


print(
    "PASS: WTDR2D = "
    "7,574 positive + 1,087 zero + 1,152 missing."
)


# ============================================================
# 16. DAY-1 / DAY-2 WEIGHT ROUTING
# ============================================================

#
# Participants with WTDRD1 = 0 have no Day 1 intake,
# therefore WTDR2D must be missing.
#

day1_zero = (
    df["WTDRD1"]
    .eq(0)
    .to_numpy(
        dtype=bool
    )
)


day2_missing = (
    df["WTDR2D"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


weight_mismatch_count = int(
    np.sum(
        day1_zero
        !=
        day2_missing
    )
)


print("\n--- DIETARY WEIGHT ROUTING ---")

print(
    "WTDRD1=0 vs WTDR2D missing mismatches:",
    weight_mismatch_count
)


if weight_mismatch_count != 0:

    raise ValueError(
        "WTDRD1 zero positions do not exactly "
        "match WTDR2D missing positions."
    )


print(
    "PASS: Day-1 / Day-2 weight routing validated."
)


# ============================================================
# 17. CDC CHECK - DR2DRSTZ
# ============================================================

EXPECTED_DR2DRSTZ = {
    1: 7453,
    2: 41,
    4: 121,
    5: 2198
}


actual_dr2drstz = (
    df["DR2DRSTZ"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_dr2drstz != EXPECTED_DR2DRSTZ:

    raise ValueError(
        "DR2DRSTZ frequencies differ.\n"
        f"Expected: {EXPECTED_DR2DRSTZ}\n"
        f"Actual:   {actual_dr2drstz}"
    )


if int(
    df["DR2DRSTZ"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Unexpected DR2DRSTZ missing values."
    )


print(
    "PASS: DR2DRSTZ matches CDC."
)


# ============================================================
# 18. CDC CHECK - DRDINT
# ============================================================

EXPECTED_DRDINT = {
    1: 1087,
    2: 7574
}


actual_drdint = (
    df["DRDINT"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_drdint != EXPECTED_DRDINT:

    raise ValueError(
        "DRDINT differs from CDC."
    )


if int(
    df["DRDINT"]
    .isna()
    .sum()
) != 1152:

    raise ValueError(
        "Unexpected DRDINT missing count."
    )


print(
    "PASS: DRDINT = "
    "1,087 Day-1-only + 7,574 two-day."
)


# ============================================================
# 19. DRDINT / WTDR2D CONSISTENCY
# ============================================================

two_day_intake = (
    df["DRDINT"]
    .eq(2)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


positive_two_day_weight = (
    df["WTDR2D"]
    .gt(0)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


mismatch_count = int(
    np.sum(
        two_day_intake
        !=
        positive_two_day_weight
    )
)


if mismatch_count != 0:

    raise ValueError(
        "DRDINT=2 does not match positive WTDR2D."
    )


day1_only = (
    df["DRDINT"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


zero_two_day_weight = (
    df["WTDR2D"]
    .eq(0)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


mismatch_count = int(
    np.sum(
        day1_only
        !=
        zero_two_day_weight
    )
)


if mismatch_count != 0:

    raise ValueError(
        "DRDINT=1 does not match WTDR2D=0."
    )


print(
    "PASS: DRDINT and WTDR2D patterns agree."
)


# ============================================================
# 20. CDC CHECK - DR2DBIH
# ============================================================

#
# Days between Day-2 intake and household interview.
#
# CDC:
#
# range    = -28 to 157
# valid    = 7,560
# missing  = 2,253
#
# Attached XPT contains 8 valid zeros.
#

s = df["DR2DBIH"]


if int(
    s.notna()
    .sum()
) != 7560:

    raise ValueError(
        "Unexpected DR2DBIH nonmissing count."
    )


if int(
    s.isna()
    .sum()
) != 2253:

    raise ValueError(
        "Unexpected DR2DBIH missing count."
    )


if int(
    s.eq(0)
    .sum()
) != 8:

    raise ValueError(
        "Unexpected DR2DBIH zero count."
    )


if s.dropna().min() != -28:

    raise ValueError(
        "Unexpected DR2DBIH minimum."
    )


if s.dropna().max() != 157:

    raise ValueError(
        "Unexpected DR2DBIH maximum."
    )


print(
    "PASS: DR2DBIH range -28 to 157 "
    "with 8 validated zeros."
)


# ============================================================
# 21. CDC CHECK - DR2TNUMF
# ============================================================

#
# Total foods/beverages reported.
#
# CDC released range:
#
#     0-47
#
# valid = 7,574
# missing = 2,239
#
# This file has exactly ONE valid zero.
#

s = df["DR2TNUMF"]


if int(
    s.notna()
    .sum()
) != 7574:

    raise ValueError(
        "Unexpected DR2TNUMF nonmissing count."
    )


if int(
    s.isna()
    .sum()
) != 2239:

    raise ValueError(
        "Unexpected DR2TNUMF missing count."
    )


if int(
    s.eq(0)
    .sum()
) != 1:

    raise ValueError(
        "Unexpected DR2TNUMF zero count."
    )


if s.dropna().min() != 0:

    raise ValueError(
        "Unexpected DR2TNUMF minimum."
    )


if s.dropna().max() != 47:

    raise ValueError(
        "Unexpected DR2TNUMF maximum."
    )


print(
    "PASS: DR2TNUMF range 0-47 "
    "including one valid zero."
)


# ============================================================
# 22. NUTRIENT COMPLETENESS CHECK
# ============================================================

#
# There are 65 energy/nutrient/component fields:
#
# DR2TKCAL ... DR2TP226
#
# Every one has:
#
#     7,453 nonmissing
#     2,360 missing
#
# These correspond to reliable Day-2 recalls.
#

nutrient_start = (
    EXPECTED_COLUMNS
    .index("DR2TKCAL")
)


nutrient_end = (
    EXPECTED_COLUMNS
    .index("DR2_300")
)


NUTRIENT_COLUMNS = (
    EXPECTED_COLUMNS[
        nutrient_start:nutrient_end
    ]
)


if len(NUTRIENT_COLUMNS) != 65:

    raise ValueError(
        "Expected exactly 65 nutrient/component fields."
    )


nutrient_failures = {}


for col in NUTRIENT_COLUMNS:

    nonmissing = int(
        df[col]
        .notna()
        .sum()
    )

    missing = int(
        df[col]
        .isna()
        .sum()
    )


    if (
        nonmissing != 7453
        or
        missing != 2360
    ):

        nutrient_failures[col] = {
            "nonmissing": nonmissing,
            "missing": missing
        }


if nutrient_failures:

    raise ValueError(
        "Unexpected nutrient missing pattern:\n"
        + str(nutrient_failures)
    )


print(
    "PASS: All 65 nutrient/component fields "
    "have 7,453 values and 2,360 missing."
)


# ============================================================
# 23. NUTRIENTS VS RELIABLE RECALL STATUS
# ============================================================

#
# Nutrient totals should exist exactly where:
#
#     DR2DRSTZ = 1
#

reliable_recall = (
    df["DR2DRSTZ"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


for col in NUTRIENT_COLUMNS:

    nutrient_nonmissing = (
        df[col]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )


    mismatch_count = int(
        np.sum(
            nutrient_nonmissing
            !=
            reliable_recall
        )
    )


    if mismatch_count != 0:

        raise ValueError(
            f"{col} nonmissing pattern does not "
            "match DR2DRSTZ=1."
        )


print(
    "PASS: All nutrient totals exactly match "
    "DR2DRSTZ=1 records."
)


# ============================================================
# 24. KEY ZERO-BEARING NUTRIENTS
# ============================================================

KEY_ZERO_COUNTS = {
    "DR2TALCO": 6721,
    "DR2TATOA": 6094,
    "DR2TB12A": 4737,
    "DR2TTHEO": 4253,
    "DR2TP184": 3548,
    "DR2TCAFF": 2242,
    "DR2TLYCO": 2118,
    "DR2TM221": 1289,
    "DR2TP226": 1104,
    "DR2TACAR": 1086
}


print("\n--- KEY NUTRIENT ZERO CHECK ---")


for col, expected_zero_count in KEY_ZERO_COUNTS.items():

    actual_zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col}: {actual_zero_count:,}"
    )


    if actual_zero_count != expected_zero_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_zero_count:,} zeros, "
            f"found {actual_zero_count:,}."
        )


print(
    "PASS: Key nutrient zero counts validated."
)


# ============================================================
# 25. CDC CHECK - DR2_300
# ============================================================

EXPECTED_DR2_300 = {
    1: 436,
    2: 6237,
    3: 903,
    9: 39
}


actual_dr2_300 = (
    df["DR2_300"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_dr2_300 != EXPECTED_DR2_300:

    raise ValueError(
        "DR2_300 frequencies differ from CDC."
    )


if int(
    df["DR2_300"]
    .isna()
    .sum()
) != 2198:

    raise ValueError(
        "Unexpected DR2_300 missing count."
    )


print(
    "PASS: DR2_300 matches CDC."
)


# ============================================================
# 26. WATER VARIABLES
# ============================================================

#
# CDC:
#
# DR2_320Z:
#     0-11520 grams
#
# DR2_330Z:
#     0-11520 grams
#
# DR2BWATZ:
#     0-8160 grams
#
# Each has:
#
#     7,574 valid
#     2,239 missing
#

WATER_CHECKS = {

    "DR2_320Z": {
        "zero": 1504,
        "max": 11520
    },

    "DR2_330Z": {
        "zero": 3879,
        "max": 11520
    },

    "DR2BWATZ": {
        "zero": 4675,
        "max": 8160
    }
}


print("\n--- WATER VARIABLE CHECK ---")


for col, info in WATER_CHECKS.items():

    s = df[col]


    nonmissing = int(
        s.notna()
        .sum()
    )


    missing = int(
        s.isna()
        .sum()
    )


    zero_count = int(
        s.eq(0)
        .sum()
    )


    minimum = float(
        s.dropna()
        .min()
    )


    maximum = float(
        s.dropna()
        .max()
    )


    print(
        f"{col}: "
        f"nonmissing={nonmissing:,}, "
        f"zero={zero_count:,}, "
        f"min={minimum}, "
        f"max={maximum}"
    )


    if nonmissing != 7574:

        raise ValueError(
            f"{col}: unexpected nonmissing count."
        )


    if missing != 2239:

        raise ValueError(
            f"{col}: unexpected missing count."
        )


    if zero_count != info["zero"]:

        raise ValueError(
            f"{col}: unexpected zero count."
        )


    if minimum != 0:

        raise ValueError(
            f"{col}: expected minimum 0."
        )


    if maximum != info["max"]:

        raise ValueError(
            f"{col}: unexpected maximum."
        )


print(
    "PASS: Day-2 water totals validated."
)


# ============================================================
# 27. WATER FIELDS VS TWO-DAY INTAKE
# ============================================================

#
# Water totals are available exactly for the
# 7,574 people with two intake days.
#

two_day_mask = (
    df["DRDINT"]
    .eq(2)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


for col in WATER_CHECKS:

    nonmissing_mask = (
        df[col]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )


    mismatch_count = int(
        np.sum(
            nonmissing_mask
            !=
            two_day_mask
        )
    )


    if mismatch_count != 0:

        raise ValueError(
            f"{col} nonmissing pattern does not "
            "match DRDINT=2."
        )


print(
    "PASS: Water fields exactly match "
    "the 7,574 two-day intake records."
)


# ============================================================
# 28. CDC CHECK - DR2TWS
# ============================================================

EXPECTED_DR2TWS = {
    1: 5297,
    2: 516,
    3: 98,
    4: 1398,
    99: 306
}


actual_dr2tws = (
    df["DR2TWS"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_dr2tws != EXPECTED_DR2TWS:

    raise ValueError(
        "DR2TWS frequencies differ from CDC.\n"
        f"Expected: {EXPECTED_DR2TWS}\n"
        f"Actual:   {actual_dr2tws}"
    )


if int(
    df["DR2TWS"]
    .isna()
    .sum()
) != 2198:

    raise ValueError(
        "Unexpected DR2TWS missing count."
    )


print(
    "PASS: DR2TWS matches CDC."
)


# ============================================================
# 29. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original pandas-decoded file contained no normal 0s.
#
# Therefore all zeros now present must exactly match
# the 50,469 validated corrections.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    "Variables containing zero:",
    len(final_zero_counts)
)

print(
    "Total zeros:",
    f"{sum(final_zero_counts.values()):,}"
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from validated "
        "DR2TOT_H correction pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "50,469 validated restorations."
)


# ============================================================
# 30. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Nothing besides:
#
#     TINY_VALUE -> 0
#
# may have changed numerically.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 85 variables match the exact "
    "expected corrected XPT values."
)


# ============================================================
# 31. EXPORT CSV
# ============================================================

#
# CRITICAL DIETARY RULE:
#
# NO:
#
#     float_format=
#
# NO:
#
#     .round()
#
# NO:
#
#     custom decimal formatting
#
# Genuine Dietary decimals must serialize naturally.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 32. RE-READ USING ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 33. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 85 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 34. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 35. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 36. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 37. .0 CHECK - INTEGER/CODE FIELDS ONLY
# ============================================================

#
# CRITICAL:
#
# DO NOT perform this test on all 85 columns.
#
# There are 50 genuine Dietary decimal fields.
#
# Legitimate examples:
#
#     WTDRD1    = 0.0
#     WTDR2D    = 0.0
#     DR2TALCO  = 0.0
#     DR2TP184  = 0.0
#     DR2_320Z  = 0.0
#
# Check only the 35 proven whole-number fields.
#

print("\n--- INTEGER/CODE COLUMN .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 35 whole-number/code fields."
)


# ============================================================
# 38. LEGITIMATE DECIMAL .0 VALUES
# ============================================================

#
# INFORMATIONAL.
#
# '.0' is NOT an error in genuine Dietary
# decimal columns.
#

print(
    "\n--- LEGITIMATE DIETARY DECIMAL .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}: {count:,}"
    )


total_decimal_dot_zero = sum(
    decimal_dot_zero_counts.values()
)


print(
    "\nTotal legitimate '.0' tokens "
    "inside decimal columns:",
    f"{total_decimal_dot_zero:,}"
)


print(
    "PASS: Dietary decimal columns were "
    "not globally stripped."
)


# ============================================================
# 39. EXACT DIETARY DECIMAL FIDELITY
# ============================================================

print(
    "\n--- EXACT DIETARY DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Dietary decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All 50 genuine Dietary decimal "
    "variables preserved exactly."
)


# ============================================================
# 40. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    "Variables containing zero:",
    len(csv_zero_counts)
)

print(
    "Total zeros:",
    f"{sum(csv_zero_counts.values()):,}"
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly matches "
    "the validated 50,469 corrections."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

#
# Check data cells, not arbitrary text.
#

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final CSV minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final CSV maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "DR2TOT_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Dietary - Total Nutrient Intakes, Day 2"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine Dietary decimal fields:",
    len(DECIMAL_COLUMNS)
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "WTDRD1 zero restorations:",
    EXPECTED_TINY_COUNTS["WTDRD1"]
)


print(
    "WTDR2D zero restorations:",
    EXPECTED_TINY_COUNTS["WTDR2D"]
)


print(
    "DR2TALCO zero restorations:",
    EXPECTED_TINY_COUNTS["DR2TALCO"]
)


print(
    "DR2TATOA zero restorations:",
    EXPECTED_TINY_COUNTS["DR2TATOA"]
)


print(
    "DR2TB12A zero restorations:",
    EXPECTED_TINY_COUNTS["DR2TB12A"]
)


print(
    "DR2_320Z zero restorations:",
    EXPECTED_TINY_COUNTS["DR2_320Z"]
)


print(
    "DR2_330Z zero restorations:",
    EXPECTED_TINY_COUNTS["DR2_330Z"]
)


print(
    "DR2BWATZ zero restorations:",
    EXPECTED_TINY_COUNTS["DR2BWATZ"]
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Dietary decimal differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)


print(
    "Legitimate '.0' in decimal fields: "
    "ALLOWED"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 DR2TOT_H XPT -> CSV
Rows: 9,813
Columns: 85
PASS: Structure = 9,813 rows x 85 columns.

--- FIELD TYPE PLAN ---
Genuine Dietary decimal fields: 50
Whole-number/code fields: 35

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 50,469

Tiny values by variable:
DR2TALCO    6721
DR2TATOA    6094
DR2TB12A    4737
DR2BWATZ    4675
DR2TTHEO    4253
DR2_330Z    3879
DR2TP184    3548
DR2TCAFF    2242
DR2TLYCO    2118
DR2_320Z    1504
DR2TM221    1289
WTDRD1      1152
DR2TP226    1104
WTDR2D      1087
DR2TACAR    1086
DR2TP205     912
DR2TCRYP     820
DR2TP225     673
DR2TS060     400
DR2TS040     365
DR2TFA       260
DR2TVD       212
DR2TS080     120
DR2TBCAR     119
DR2TCHOL     108
DR2TP204      99
DR2TLZ        92
DR2TFIBE      85
DR2TRET       82
DR2TFF        74
DR2TM201      54
DR2TVC        51
DR2TS100      44
DR2TM161      37
DR2TVB12      33
DR2TS120      27
DR2TVARA      23
DR2TS140      15
DR2TVK        15
DR2TS180    